# 🚀 Fine-Tuning de LLM para Arquitetura Angular Corporativa (FPFtech)

Este notebook detalha a implementação e execução de um pipeline de **Fine-Tuning Supervisionado (SFT)** utilizando a técnica **QLoRA (Quantized Low-Rank Adaptation)**. O objetivo central é especializar o modelo **Qwen2.5-Coder-3B** na geração de componentes Angular que aderem rigorosamente ao padrão `BaseComponent` da FPFtech.

### Escopo Técnico:
1. **Especialização Arquitetural**: Injeção de conhecimento sobre herança de classes e padrões de injeção corporativos.
2. **Otimização de Recursos**: Uso de kernels de GPU otimizados via biblioteca Unsloth para viabilizar o treino em hardware com baixa VRAM.
3. **Alinhamento de Saída**: Refino da capacidade do modelo em fornecer saídas estritamente em blocos de código Markdown.

## 1. Configuração do Ecossistema Técnico

O ambiente utiliza o **Unsloth**, uma biblioteca que implementa kernels Triton customizados para substituir as implementações padrão do PyTorch. Isso resulta em uma redução de aproximadamente 70% no consumo de memória e aumento de 2x na velocidade de treinamento.

*   **Transformers (Hugging Face)**: Framework base para manipulação de modelos e tokenizadores.
*   **TRL (Transformer Reinforcement Learning)**: Utilizada aqui especificamente por sua classe `SFTTrainer`, que simplifica o ciclo de treinamento supervisionado.
*   **BNB (bitsandbytes)**: Responsável pela quantização de 4-bits do modelo base.

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Hiperparâmetros globais de arquitetura
model_id = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
max_seq_length = 2048
load_in_4bit = True

print(f"GPU detectada: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma'}")

## 2. Inicialização do Modelo e Configuração do LoRA

Nesta etapa, carregamos o modelo quantizado e aplicamos os adaptadores LoRA. 

### Definições dos Hiperparâmetros de Adaptação:
*   **Rank (r=16)**: Define a dimensionalidade das matrizes de baixa ordem inseridas no modelo. Um rank de 16 é escolhido para equilibrar a capacidade de memorizar estruturas sintáticas rígidas (como TS/Angular) sem sobrecarregar a memória VRAM.
*   **Alpha (16)**: Atua como um fator de escala para a contribuição do LoRA nos pesos originais. A manutenção da proporção 1:1 com o Rank visa estabilidade na convergência.
*   **Target Modules**: Os adaptadores são injetados em **todas as camadas lineares**, incluindo as projeções de Atenção (`q`, `k`, `v`, `o`) e as camadas de MLP (`gate`, `up`, `down`). Isso garante que o modelo aprenda tanto a forma (estrutura) quanto a função (lógica operacional) dos componentes FPFtech.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout zero maximiza o sinal de aprendizado em datasets de alta qualidade
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 3. Pré-processamento do Dataset Conversacional

O dataset utiliza o formato JSONL com o campo `messages` seguindo a arquitetura de chat. 

### Processo de Tokenização:
Utilizamos o `apply_chat_template` nativo do Qwen. Ele insere tokens delimitadores especiais (como `<|im_start|>` e `<|im_end|>`) que permitem ao modelo distinguir claramente entre as instruções do sistema, as requisições do usuário e as respostas do assistente. O **System Prompt** define as restrições arquiteturais (ex: obrigatoriedade do `Injector` e herança de `BaseComponent`).

In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["messages"]
    texts = []
    for messages in instructions:
        text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return { "text" : texts }

dataset = load_dataset("json", data_files="../../data/datasets/augmented/augmented_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

print(f"Quantidade de exemplos para treino: {len(dataset)}")

## 4. Configuração do Treinamento Supervisionado (SFT)

A configuração do `TrainingArguments` define a estratégia matemática de convergência:

*   **Gradient Accumulation Steps (8)**: Como o batch size físico é 1 (para evitar estouro de VRAM), acumulamos os gradientes de 8 exemplos consecutivos antes de atualizar os pesos. Isso simula um **Global Batch Size de 8**, conferindo estabilidade estatística ao cálculo do gradiente.
*   **Cosine LR Scheduler**: A taxa de aprendizado decai seguindo uma função de cosseno. Isso permite um aprendizado agressivo no início (exploração) e um refinamento infinitesimal ao final (convergência de alta precisão).
*   **BF16 (Bfloat16)**: Formato de ponto flutuante de 16 bits que preserva o range dinâmico do FP32, essencial para evitar instabilidades em modelos de grande escala.

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 5e-5,
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        output_dir = "../../data/output_qlora_results",
    ),
)

# Comentado para visualização passiva:
# trainer.train()

## 5. Auditoria de Métricas e Estabilidade

Para validar a eficácia do treinamento, analisamos duas métricas principais:

1.  **Loss (Erro)**: Mede a entropia cruzada entre a predição do modelo e o token real do dataset. A redução consistente indica que o modelo está aprendendo a linguagem alvo.
2.  **Grad Norm (Norma L2 do Gradiente)**: Mede a magnitude total das atualizações de pesos. Valores estáveis (geralmente < 1.0) indicam que o modelo não está sofrendo de "explosão de gradientes", mantendo a integridade matemática dos pesos originais.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

metrics_path = "../../data/output_qlora_v2/metrics/training_metrics_all_steps.json"
if os.path.exists(metrics_path):
    df = pd.read_json(metrics_path)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss Plot
    ax1.plot(df['epoch'], df['loss'], color='dodgerblue')
    ax1.set_title('Convergência de Loss')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Erro (Loss)')
    ax1.grid(True, linestyle='--')
    
    # Grad Norm Plot
    ax2.plot(df['epoch'], df['grad_norm'], color='mediumorchid')
    ax2.set_title('Estabilidade (Grad Norm)')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Magnitude do Gradiente')
    ax2.grid(True, linestyle='--')
    
    plt.tight_layout()
    plt.show()
else:
    print("Métricas reais não encontradas no sistema de arquivos.")

## 6. Inferência e Avaliação Comparativa

O modo de inferência desativa as operações de dropout e ativa a cache de KV para acelerar a geração. O prompt do sistema injetado aqui é o mesmo utilizado no treino, garantindo que o modelo entre no estado mental de "FPFtech Angular Coder".

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model) # Otimização de tempo de execução

test_prompt = "Crie um componente que estenda BaseComponent para gerenciar o estoque de produtos"
messages = [
    {"role": "system", "content": "Você é o FPFtech Angular Coder. Gere APENAS código TypeScript Angular seguindo o padrão BaseComponent da FPFtech."},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.1) # Baixa temperatura para maior determinismo técnico
response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

print("### RESPOSTA TÉCNICA GERADA ###")
print(response)